In [1]:
from pathlib import Path
import time
import json
from tqdm.auto import tqdm
import gc

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

In [2]:
!pip install tabpfn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/827.5 kB ? eta -:--:--


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━ 675.8/827.5 kB 20.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.5/827.5 kB 15.3 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/595.9 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 33.3 MB/s eta 0:00:00


In [3]:
final_data_path = Path("../input/datasets/aniaandruszkiewicz/thesis-sub")
final_data_path_output = Path("working")
features_dir = final_data_path
results_dir = final_data_path_output / "modeling_results" / "tabpfn"
results_dir.mkdir(parents=True, exist_ok=True)

layer_files = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]

target_col = "dep_del15"
non_feature_cols = ["flight_id", "year", target_col]

datasets = {}
for name in layer_files:
    key = f"{name}_subsampled"
    df = pd.read_parquet(features_dir / f"{key}_features.parquet")
    datasets[key] = df
    print(f"{key}: {df.shape}")

layer_a_bts_subsampled: (6966559, 28)


layer_b_bts_meteostat_subsampled: (6966559, 32)


layer_c_bts_gdelt_subsampled: (6966559, 43)


layer_d_bts_meteostat_gdelt_subsampled: (6966559, 47)


In [4]:
def get_train_test(key: str):
    """Split one dataset by year and drop non-feature columns from X."""
    df = datasets[key]
    train = df[df["year"] == 2024]
    test = df[df["year"] == 2025]

    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    X_train, y_train = train[feature_cols], train[target_col]
    X_test, y_test = test[feature_cols], test[target_col]
    return X_train, y_train, X_test, y_test

In [5]:
def get_train_val_test(key: str, val_size=0.15, random_state=42):
    df = datasets[key]
    train_full = df[df["year"] == 2024]
    test = df[df["year"] == 2025]

    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    train, val = train_test_split(
        train_full, test_size=val_size, stratify=train_full[target_col], random_state=random_state
    )

    X_train, y_train = train[feature_cols], train[target_col]
    X_val, y_val = val[feature_cols], val[target_col]
    X_test, y_test = test[feature_cols], test[target_col]
    return X_train, y_train, X_val, y_val, X_test, y_test

In [6]:
beta=1.2

def tune_threshold(y_true, oob_proba, beta=beta):
    precision, recall, thresholds = precision_recall_curve(y_true, oob_proba)
    fbeta_scores = (
        (1 + beta**2) * precision * recall
        / (beta**2 * precision + recall + 1e-12)
    )
    best_idx = np.nanargmax(fbeta_scores[:-1])
    return thresholds[best_idx]

In [7]:
def predict_proba_chunked(clf, X_test, batch_size=50_000, desc="predicting batches"):
    n = len(X_test)
    proba = np.empty(n, dtype=np.float64)

    for start in tqdm(range(0, n, batch_size), desc=desc):
        end = min(start + batch_size, n)
        proba[start:end] = clf.predict_proba(X_test.iloc[start:end])[:, 1]

    return proba

In [8]:
import os
os.environ["TABPFN_TOKEN"] = "tabpfn_sk_fOIuKkH8Vz_XIP5KqKN4SuqPjVEDY0D4MoX20NdfkRE"

In [9]:
from tabpfn import TabPFNClassifier

X_warm, y_warm, _, _ = get_train_test("layer_a_bts_subsampled")
clf = TabPFNClassifier()
clf.fit(X_warm.iloc[:100], y_warm.iloc[:100])
print("license accepted, checkpoint downloaded")

tabpfn-v3.5-20260909.safetensors:   0%|          | 0.00/876M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

license accepted, checkpoint downloaded


In [10]:
search_layer = "layer_d_bts_meteostat_gdelt_subsampled"
search_train_sample_size = 200_000
search_probe_size = 50_000
search_batchsize = 20_000

n_estimators_grid = [1, 2, 4]
subsample_samples_grid = [10_000, 20_000, 40_000]

X_train_s, y_train_s, X_val_s, y_val_s, X_test_s, y_test_s = get_train_val_test(search_layer)
n_full_test = len(X_test_s)

if len(X_train_s) > search_train_sample_size:
    X_train_s, _, y_train_s, _ = train_test_split(
        X_train_s, y_train_s,
        train_size=search_train_sample_size,
        stratify=y_train_s, random_state=42,
    )

# small, stratified probe slice of the real test set for scoring + timing
_, X_probe, _, y_probe = train_test_split(
    X_test_s, y_test_s,
    test_size=search_probe_size,
    stratify=y_test_s, random_state=42,
)

search_results = []

for n_est in n_estimators_grid:
    for sub_samples in subsample_samples_grid:
        try:
            t0 = time.time()
            clf = TabPFNClassifier(
                device="cuda",
                n_estimators=n_est,
                inference_config={"SUBSAMPLE_SAMPLES": sub_samples},
            )
            clf.fit(X_train_s, y_train_s)
            fit_time = time.time() - t0

            t1 = time.time()
            probe_proba = predict_proba_chunked(
                clf, X_probe, batch_size=search_batchsize,
                desc=f"probe n_est={n_est} sub={sub_samples}",
            )
            predict_time_probe = time.time() - t1

            # extrapolate probe predict time to the full test set for this layer
            predict_time_full_est = predict_time_probe * (n_full_test / len(X_probe))
            total_time_full_est = fit_time + predict_time_full_est

            auc = roc_auc_score(y_probe, probe_proba)
            f1_default = f1_score(y_probe, (probe_proba >= 0.5).astype(int))

            search_results.append({
                "n_estimators": n_est,
                "subsample_samples": sub_samples,
                "probe_auc_roc": auc,
                "probe_f1_at_0.5": f1_default,
                "fit_seconds": fit_time,
                "predict_seconds_probe": predict_time_probe,
                "est_predict_seconds_full_layer": predict_time_full_est,
                "est_total_seconds_full_layer": total_time_full_est,
            })

            print(f"n_estimators={n_est}, subsample_samples={sub_samples} -> "
                  f"AUC={auc:.4f}, F1@0.5={f1_default:.4f}, "
                  f"est. full-layer time={total_time_full_est/60:.1f} min")

        except RuntimeError as e:
            print(f"n_estimators={n_est}, subsample_samples={sub_samples} FAILED: {e}")
        finally:
            del clf
            gc.collect()
            torch.cuda.empty_cache()

search_df = pd.DataFrame(search_results).sort_values("probe_auc_roc", ascending=False)
search_df.to_csv(results_dir / "param_search_fm.csv", index=False)
search_df

probe n_est=1 sub=10000:   0%|          | 0/3 [00:00<?, ?it/s]

n_estimators=1, subsample_samples=10000 -> AUC=0.6725, F1@0.5=0.0787, est. full-layer time=30.4 min


probe n_est=1 sub=20000:   0%|          | 0/3 [00:00<?, ?it/s]

n_estimators=1, subsample_samples=20000 -> AUC=0.6739, F1@0.5=0.0835, est. full-layer time=58.5 min


probe n_est=1 sub=40000:   0%|          | 0/3 [00:00<?, ?it/s]

...

probe n_est=4 sub=20000:   0%|          | 0/3 [00:00<?, ?it/s]

n_estimators=4, subsample_samples=20000 -> AUC=0.6817, F1@0.5=0.0532, est. full-layer time=251.4 min


probe n_est=4 sub=40000:   0%|          | 0/3 [00:00<?, ?it/s]

n_estimators=4, subsample_samples=40000 -> AUC=0.6835, F1@0.5=0.0721, est. full-layer time=624.5 min


,n_estimators,subsample_samples,probe_auc_roc,probe_f1_at_0.5,fit_seconds,predict_seconds_probe,est_predict_seconds_full_layer,est_total_seconds_full_layer
8,4,40000,0.683469,0.072097,2.767427,313.993255,37469.185676,37471.953103
7,4,20000,0.681705,0.053162,2.630781,126.369440,15079.814429,15082.445210
5,2,40000,0.681438,0.064633,2.520073,157.071866,18743.571062,18746.091135
6,4,10000,0.680669,0.038125,2.593890,62.326339,7437.475607,7440.069497
2,1,40000,0.679819,0.075878,2.950753,79.340392,9467.782612,9470.733366
4,2,20000,0.679346,0.063102,2.468844,63.577855,7586.820472,7589.289316
3,2,10000,0.675429,0.045518,2.941388,30.876853,3684.571273,3687.512661
1,1,20000,0.673888,0.083460,2.876667,29.386321,3506.704312,3509.580979
0,1,10000,0.672460,0.078713,2.566231,15.272289,1822.460301,1825.026532


In [11]:
results = {}
layer_probas = {}

tabpfn_n_estimators = 2
tabpfn_subsample_samples = 20_000
predict_batch_size = 20_000

for name in tqdm(layer_files, desc="TabPFN layers"):
    key = f"{name}_subsampled"
    X_train, y_train, X_val, y_val, X_test, y_test = get_train_val_test(key)

    t0 = time.time()
    clf = TabPFNClassifier(
        device="cuda",
        n_estimators=tabpfn_n_estimators,
        inference_config={"SUBSAMPLE_SAMPLES": tabpfn_subsample_samples},
    )
    clf.fit(X_train, y_train)

    val_proba = predict_proba_chunked(clf, X_val, batch_size=predict_batch_size, desc="predicting val batches")
    threshold = tune_threshold(y_val, val_proba, beta=beta)

    y_proba = predict_proba_chunked(clf, X_test, batch_size=predict_batch_size, desc="predicting test batches")
    y_pred = (y_proba >= threshold).astype(int)
    elapsed = time.time() - t0

    metrics = {
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
        "mcc": matthews_corrcoef(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "threshold": threshold,
        "n_estimators": tabpfn_n_estimators,
        "subsample_samples": tabpfn_subsample_samples,
        "train_seconds": elapsed,
    }
    results[key] = metrics

    # checkpoint after every layer so a disconnect doesn't cost the whole run
    with open(results_dir / f"{key}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    np.save(results_dir / f"{key}_y_proba.npy", y_proba)
    np.save(results_dir / f"{key}_y_pred.npy", y_pred)

    # cache for beta sweep
    layer_probas[key] = {
        "y_val": y_val,
        "val_proba": val_proba,
        "y_test": y_test,
        "test_proba": y_proba,
    }

    print(f"{key}: {metrics}")

    del clf, X_train, y_train, X_val, X_test, y_pred
    gc.collect()
    torch.cuda.empty_cache()

TabPFN layers:   0%|          | 0/4 [00:00<?, ?it/s]

predicting val batches:   0%|          | 0/8 [00:00<?, ?it/s]

predicting test batches:   0%|          | 0/299 [00:00<?, ?it/s]

layer_a_bts_subsampled: {'f1': 0.4218822523971767, 'roc_auc': np.float64(0.6738687147404723), 'pr_auc': np.float64(0.3589054483668561), 'mcc': np.float64(0.20608199732906926), 'precision': 0.3083834107038896, 'recall': 0.6675822953566043, 'threshold': np.float64(0.187743678689003), 'n_estimators': 2, 'subsample_samples': 20000, 'train_seconds': 6270.710880994797}


predicting val batches:   0%|          | 0/8 [00:00<?, ?it/s]

...

predicting test batches:   0%|          | 0/299 [00:00<?, ?it/s]

layer_c_bts_gdelt_subsampled: {'f1': 0.4218470557182875, 'roc_auc': np.float64(0.6734916022916342), 'pr_auc': np.float64(0.35546759999766414), 'mcc': np.float64(0.20543423089660048), 'precision': 0.306181667013095, 'recall': 0.6779569488316721, 'threshold': np.float64(0.1866697520017624), 'n_estimators': 2, 'subsample_samples': 20000, 'train_seconds': 6818.791967630386}


predicting val batches:   0%|          | 0/8 [00:00<?, ?it/s]

predicting test batches:   0%|          | 0/299 [00:00<?, ?it/s]

layer_d_bts_meteostat_gdelt_subsampled: {'f1': 0.42857786354674826, 'roc_auc': np.float64(0.6817631428470725), 'pr_auc': np.float64(0.36363344573261436), 'mcc': np.float64(0.21923535931525626), 'precision': 0.320680747409989, 'recall': 0.6458986942894532, 'threshold': np.float64(0.1962883174419403), 'n_estimators': 2, 'subsample_samples': 20000, 'train_seconds': 6969.179116487503}


In [12]:
with open(results_dir / "all_layers_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

pd.DataFrame(results).T

,f1,roc_auc,pr_auc,mcc,precision,recall,threshold,n_estimators,subsample_samples,train_seconds
layer_a_bts_subsampled,0.421882,0.673869,0.358905,0.206082,0.308383,0.667582,0.187744,2.0,20000.0,6270.710881
layer_b_bts_meteostat_subsampled,0.430450,0.684213,0.369190,0.220576,0.315344,0.677890,0.186673,2.0,20000.0,6413.284578
layer_c_bts_gdelt_subsampled,0.421847,0.673492,0.355468,0.205434,0.306182,0.677957,0.186670,2.0,20000.0,6818.791968
layer_d_bts_meteostat_gdelt_subsampled,0.428578,0.681763,0.363633,0.219235,0.320681,0.645899,0.196288,2.0,20000.0,6969.179116


In [13]:
summary_df = pd.DataFrame(results).T
summary_df.index.name = "layer"
summary_df = summary_df.reset_index()

# match RF summary's column naming/order where possible
summary_df = summary_df.rename(columns={"roc_auc": "auc_roc", "pr_auc": "auc_pr", "train_seconds": "fit_time_s"})
summary_df = summary_df[["layer", "f1", "precision", "recall", "auc_roc", "auc_pr", "mcc", "threshold", "n_estimators", "subsample_samples", "fit_time_s"]]

summary_df.to_csv(results_dir / "fm_subsampled_summary.csv", index=False)
summary_df

,layer,f1,precision,recall,auc_roc,auc_pr,mcc,threshold,n_estimators,subsample_samples,fit_time_s
0,layer_a_bts_subsampled,0.421882,0.308383,0.667582,0.673869,0.358905,0.206082,0.187744,2.0,20000.0,6270.710881
1,layer_b_bts_meteostat_subsampled,0.430450,0.315344,0.677890,0.684213,0.369190,0.220576,0.186673,2.0,20000.0,6413.284578
2,layer_c_bts_gdelt_subsampled,0.421847,0.306182,0.677957,0.673492,0.355468,0.205434,0.186670,2.0,20000.0,6818.791968
3,layer_d_bts_meteostat_gdelt_subsampled,0.428578,0.320681,0.645899,0.681763,0.363633,0.219235,0.196288,2.0,20000.0,6969.179116


In [14]:
from sklearn.inspection import permutation_importance

importance_sample_size = 3_000  # small on purpose

importance_results = {}

for name in tqdm(layer_files, desc="Feature importance"):
    key = f"{name}_subsampled"
    X_train, y_train, X_val, y_val, X_test, y_test = get_train_val_test(key)

    #same config used in the main loop
    clf = TabPFNClassifier(
        device="cuda",
        n_estimators=tabpfn_n_estimators,
        inference_config={"SUBSAMPLE_SAMPLES": tabpfn_subsample_samples},
    )
    clf.fit(X_train, y_train)

    # small held-out sample for importance — keep this small, cost scales with n_features x n_repeats
    sample_idx = X_test.sample(n=min(importance_sample_size, len(X_test)), random_state=42).index
    X_sample = X_test.loc[sample_idx]
    y_sample = y_test.loc[sample_idx]

    t0 = time.time()
    perm_result = permutation_importance(
        clf, X_sample, y_sample,
        scoring="roc_auc",
        n_repeats=5,
        random_state=42,
        n_jobs=1,
    )
    elapsed = time.time() - t0

    importance_df = pd.DataFrame({
        "feature": X_sample.columns,
        "importance_mean": perm_result.importances_mean,
        "importance_std": perm_result.importances_std,
    }).sort_values("importance_mean", ascending=False)

    importance_df.to_csv(results_dir / f"{key}_feature_importance.csv", index=False)
    importance_results[key] = importance_df

    print(f"{key}: computed in {elapsed:.1f}s")
    print(importance_df.head(10))

    del clf, X_train, y_train, X_val, X_test, X_sample, y_sample
    gc.collect()
    torch.cuda.empty_cache()

Feature importance:   0%|          | 0/4 [00:00<?, ?it/s]

layer_a_bts_subsampled: computed in 1549.3s
             feature  importance_mean  importance_std
6           hour_sin         0.105730        0.009638
3          month_cos         0.019897        0.004084
7           hour_cos         0.007344        0.001140
9            dest_te         0.005933        0.001638
0   crs_elapsed_time         0.005077        0.001947
8          origin_te         0.004948        0.000904
4            dow_sin         0.003980        0.001178
23        carrier_WN         0.002991        0.001415
1           distance         0.002734        0.000477
22        carrier_UA         0.002230        0.000124


layer_b_bts_meteostat_subsampled: computed in 1840.6s
             feature  importance_mean  importance_std
10          hour_sin         0.110821        0.009004
7          month_cos         0.013780        0.003905
2      temp_mean_12h         0.007876        0.003178
12         origin_te         0.005942        0.001302
13           dest_te         0.005170        0.001614
11          hour_cos         0.004231        0.001212
0   crs_elapsed_time         0.003523        0.001907
8            dow_sin         0.003407        0.001398
27        carrier_WN         0.003180        0.001459
5      cldc_mean_12h         0.003008        0.001446


layer_c_bts_gdelt_subsampled: computed in 2709.0s
                      feature  importance_mean  importance_std
21                   hour_sin         0.107237        0.010356
18                  month_cos         0.015587        0.003477
24                    dest_te         0.004763        0.001516
23                  origin_te         0.003986        0.000733
22                   hour_cos         0.003736        0.001396
38                 carrier_WN         0.002759        0.001111
19                    dow_sin         0.002435        0.000649
12  gdelt_assault_count_lag7d         0.001942        0.001210
30                 carrier_F9         0.001878        0.000334
1                    distance         0.001649        0.000768
